# Deep Learning: Evaluasi & Eksperimen Perbaikan (Anggota 4)
**Klasifikasi Jenis Sampah Berbasis Citra Menggunakan CNN dengan Data Augmentation dan Teknik Regularisasi**

## Persiapan di Google Drive (sekali saja)
Buat folder **`UTS_DeepLearning`** di *My Drive*, lalu upload file berikut ke dalamnya:

```
My Drive/
└── UTS_DeepLearning/
    ├── CNN_Sampah.py                (dari Anggota 3)
    ├── evaluasi_model.py            (Anggota 4)
    ├── eksperimen_perbaikan.py      (Anggota 4)
    ├── datasett_preprocessed.zip    (hasil Anggota 2, berbentuk ZIP)
    └── cnn_training_output/         (dari Anggota 3)
        ├── best_cnn_sampah.keras
        └── training_log.csv
```

## Cara pakai
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Jalankan sel **Langkah 1 sampai 4** secara berurutan (wajib setiap kali membuka notebook / setelah sesi terputus).
3. Jalankan eksperimen **satu sel per eksperimen**. Hasil tersimpan otomatis di `My Drive/UTS_DeepLearning/hasil_eksperimen/`.

> Jika sesi terputus saat eksperimen berjalan: jalankan ulang Langkah 1 sampai 4, lalu ulangi sel eksperimen yang terputus. Eksperimen yang sudah selesai tidak perlu diulang.

## Langkah 1: Cek GPU

In [ ]:
import tensorflow as tf
gpu = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpu if gpu else "TIDAK ADA. Aktifkan di Runtime > Change runtime type > T4 GPU")
!nvidia-smi -L

## Langkah 2: Hubungkan Google Drive & cek file

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_DIR = Path("/content/drive/MyDrive/UTS_DeepLearning")   # ubah jika nama folder berbeda

wajib = [
    "CNN_Sampah.py",
    "evaluasi_model.py",
    "eksperimen_perbaikan.py",
    "datasett_preprocessed.zip",
    "cnn_training_output/best_cnn_sampah.keras",
    "cnn_training_output/training_log.csv",
]
kurang = [f for f in wajib if not (DRIVE_DIR / f).exists()]
for f in wajib:
    print(("OK      " if f not in kurang else "TIDAK ADA ") + f)
if kurang:
    raise FileNotFoundError(f"Upload dulu file berikut ke {DRIVE_DIR}: {kurang}")
print("\nSemua file lengkap.")

## Langkah 3: Siapkan folder kerja
Dataset diekstrak ke disk lokal Colab (lebih cepat dibaca), sedangkan folder `hasil_eksperimen` dihubungkan ke Google Drive agar hasil tidak hilang.

In [ ]:
import os, shutil, zipfile

PROJ = Path("/content/proj")
PROJ.mkdir(exist_ok=True)

# 1) salin skrip & output Anggota 3 (selalu disalin ulang agar versi terbaru dari Drive terpakai)
for f in ["CNN_Sampah.py", "evaluasi_model.py", "eksperimen_perbaikan.py"]:
    shutil.copy(DRIVE_DIR / f, PROJ / f)
shutil.copytree(DRIVE_DIR / "cnn_training_output", PROJ / "cnn_training_output", dirs_exist_ok=True)

# Perbaikan otomatis: pastikan evaluasi_model.py bisa membuka model dari versi Keras berbeda
_FUNGSI = '''
def muat_model(path):
    try:
        return tf.keras.models.load_model(str(path), compile=False)
    except (TypeError, ValueError):
        print("[INFO] Versi Keras berbeda. Membangun ulang arsitektur dari CNN_Sampah.build_cnn() dan memuat bobotnya...")
        _, _, _, class_names = base.collect_splits(base.DATASET_DIR)
        model = base.build_cnn(num_classes=len(class_names))
        model.load_weights(str(path))
        print("[INFO] Bobot berhasil dimuat.")
        return model


def main() -> None:'''
_p = PROJ / "evaluasi_model.py"
_s = _p.read_text(encoding="utf-8")
if "def muat_model" not in _s:
    _s = _s.replace("\ndef main() -> None:", _FUNGSI, 1)
    _s = _s.replace("model = tf.keras.models.load_model(args.model, compile=False)", "model = muat_model(args.model)", 1)
    _p.write_text(_s, encoding="utf-8")

# 2) ekstrak dataset (hanya jika belum)
DATA = PROJ / "datasett_preprocessed"
if not DATA.exists():
    tmp = Path("/content/_ekstrak")
    print("Mengekstrak dataset... (1-3 menit)")
    with zipfile.ZipFile(DRIVE_DIR / "datasett_preprocessed.zip") as z:
        z.extractall(tmp)
    # cari folder yang berisi folder-folder kelas (menangani ZIP bertingkat)
    kandidat = [p for p in [tmp, *tmp.rglob("*")] if p.is_dir()
                and sum(1 for c in p.iterdir() if c.is_dir()) >= 5
                and not any(c.is_file() for c in p.iterdir())]
    sumber = min(kandidat, key=lambda p: len(p.parts))
    shutil.move(str(sumber), str(DATA))
    shutil.rmtree(tmp, ignore_errors=True)

# 3) hasil_eksperimen -> Google Drive
HASIL_DRIVE = DRIVE_DIR / "hasil_eksperimen"
HASIL_DRIVE.mkdir(exist_ok=True)
link = PROJ / "hasil_eksperimen"
if not link.exists():
    os.symlink(HASIL_DRIVE, link)

os.chdir(PROJ)
print("Folder kerja :", PROJ)
print("Kelas        :", sorted(p.name for p in DATA.iterdir() if p.is_dir()))
print("Hasil disimpan ke:", HASIL_DRIVE)

## Langkah 4: Verifikasi pembagian data
Pembagian train/val/test harus **sama persis** dengan saat Anggota 3 melatih baseline. Jika tidak sama, test set bisa berisi gambar training sehingga hasil evaluasi tidak valid.

In [ ]:
import sys
sys.path.insert(0, str(PROJ))
import CNN_Sampah as base

# angka dari log training Anggota 3
DIHARAPKAN = {
    "cardboard": (398, 49, 51), "Food Organics": (328, 41, 42), "glass": (468, 58, 60),
    "metal": (453, 56, 58), "Miscellaneous Trash": (396, 49, 50), "paper": (555, 69, 70),
    "Plastic": (736, 92, 93), "Textile Trash": (254, 31, 33), "trash": (109, 13, 15),
    "Vegetation": (347, 43, 44),
}
tr, va, te, kelas = base.collect_splits(base.DATASET_DIR)
cocok = True
print(f"{'Kelas':22s}{'train':>7s}{'val':>6s}{'test':>6s}   status")
for i, k in enumerate(kelas):
    n = (sum(l == i for _, l in tr), sum(l == i for _, l in va), sum(l == i for _, l in te))
    ok = DIHARAPKAN.get(k) == n
    cocok &= ok
    print(f"{k:22s}{n[0]:7d}{n[1]:6d}{n[2]:6d}   {'OK' if ok else 'BERBEDA, harap: ' + str(DIHARAPKAN.get(k))}")
print("\nPEMBAGIAN DATA SAMA DENGAN BASELINE" if cocok else
      "\nPERINGATAN: pembagian data BERBEDA. Pastikan ZIP dataset sama dengan yang dipakai Anggota 3.")

In [ ]:
# Fungsi bantu untuk menampilkan gambar hasil di notebook
from IPython.display import Image, display, Markdown
def tampilkan(nama, files=("diagnosis_training.txt", "training_curves.png", "confusion_matrix.png",
                           "f1_per_kelas.png", "contoh_salah_klasifikasi.png", "gradcam.png")):
    folder = PROJ / "hasil_eksperimen" / nama
    display(Markdown(f"### Hasil: `{nama}`"))
    for f in files:
        p = folder / f
        if not p.exists():
            continue
        display(Markdown(f"**{f}**"))
        if p.suffix == ".png":
            display(Image(filename=str(p), width=1100))
        else:
            print(p.read_text(encoding="utf-8"))

---
## Bagian A: Evaluasi model baseline (Anggota 3)
Waktu: ± 1 menit.

In [ ]:
!python evaluasi_model.py

In [ ]:
tampilkan("E0_baseline")
print(open("hasil_eksperimen/E0_baseline/akurasi_per_sumber.csv").read())

---
## Bagian B: Eksperimen perbaikan
### B0. Uji cepat (cek skrip berjalan, ± 3 menit)
Hasil uji cepat disimpan di folder berawalan `_uji_` dan **tidak** ikut tabel perbandingan.

In [ ]:
!python eksperimen_perbaikan.py --exp E2 --uji-cepat

### B1. E2: Transfer learning MobileNetV2 + fine-tuning
Eksperimen utama (perkiraan dengan GPU T4: ± 10–25 menit).

In [ ]:
!python eksperimen_perbaikan.py --exp E2

In [ ]:
tampilkan("E2_mobilenetv2")

### B2. Studi ablasi: pengaruh augmentasi & regularisasi
Satu sel per eksperimen agar aman jika sesi terputus (perkiraan ± 5–10 menit per sel).

In [ ]:
!python eksperimen_perbaikan.py --exp A1

In [ ]:
!python eksperimen_perbaikan.py --exp A2

In [ ]:
!python eksperimen_perbaikan.py --exp A3

In [ ]:
!python eksperimen_perbaikan.py --exp A4

In [ ]:
for n in ["A1_tanpa_aug_tanpa_reg", "A2_aug_saja", "A3_reg_saja", "A4_aug_dan_reg"]:
    tampilkan(n, files=("training_curves.png", "confusion_matrix.png"))

### B3. E1: CNN baseline yang distabilkan (dari nol)
Arsitektur sama dengan Anggota 3, training diperbaiki. Paling lama (perkiraan dengan GPU: ± 30–60 menit).

In [ ]:
!python eksperimen_perbaikan.py --exp E1

In [ ]:
tampilkan("E1_cnn_stabil")

### B4. (Opsional) E3: EfficientNetB0

In [ ]:
# !python eksperimen_perbaikan.py --exp E3

---
## Bagian C: Ringkasan perbandingan (untuk laporan)
Menghasilkan `perbandingan_eksperimen.md` (tabel siap tempel), `perbandingan_eksperimen.csv`, dan dua grafik perbandingan.

In [ ]:
!python eksperimen_perbaikan.py --ringkasan

In [ ]:
display(Markdown(open("hasil_eksperimen/perbandingan_eksperimen.md", encoding="utf-8").read()))
display(Image(filename="hasil_eksperimen/perbandingan_eksperimen.png", width=1000))
display(Image(filename="hasil_eksperimen/f1_per_kelas_perbandingan.png", width=1100))

---
## (Opsional) Unduh semua hasil sebagai ZIP
Semua hasil sudah ada di Google Drive. Sel ini hanya untuk mengunduh satu file ZIP ke laptop.

In [ ]:
shutil.make_archive("/content/hasil_eksperimen", "zip", root_dir=str(HASIL_DRIVE))
from google.colab import files
files.download("/content/hasil_eksperimen.zip")